# AdaBoost Classifier - Weather Rain Prediction

## 1. Load Weather Dataset
Load the weather dataset from the AdaBoost classifier folder and verify the file path.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

data_path = Path("weather_forecast_data.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path.resolve()}")

df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.head()

Shape: (2500, 6)


,Temperature,Humidity,Wind_Speed,Cloud_Cover,Pressure,Rain
0,23.720338,89.592641,7.335604,50.501694,1032.378759,rain
1,27.879734,46.489704,5.952484,4.990053,992.614190,no rain
2,25.069084,83.072843,1.371992,14.855784,1007.231620,no rain
3,23.622080,74.367758,7.050551,67.255282,982.632013,rain
4,20.591370,96.858822,4.643921,47.676444,980.825142,no rain


## 2. Inspect and Clean Data
Handle missing values, fix data types, and normalize target labels.

In [2]:
df.info()

numeric_cols = ["Temperature", "Humidity", "Wind_Speed", "Cloud_Cover", "Pressure"]

if "Rain" not in df.columns:
    raise ValueError("Expected target column 'Rain' not found.")

df["Rain"] = df["Rain"].astype(str).str.strip().str.lower()
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=numeric_cols + ["Rain"]).reset_index(drop=True)
print("After cleaning:", df.shape)
print(df["Rain"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Temperature  2500 non-null   float64
 1   Humidity     2500 non-null   float64
 2   Wind_Speed   2500 non-null   float64
 3   Cloud_Cover  2500 non-null   float64
 4   Pressure     2500 non-null   float64
 5   Rain         2500 non-null   object 
dtypes: float64(5), object(1)
memory usage: 117.3+ KB
After cleaning: (2500, 6)
Rain
no rain    2186
rain        314
Name: count, dtype: int64


## 3. Feature Engineering
Select features and encode the target labels.

In [3]:
X = df[numeric_cols].copy()
y = df["Rain"].map({"no rain": 0, "rain": 1})

if y.isna().any():
    unknown = df.loc[y.isna(), "Rain"].unique().tolist()
    raise ValueError(f"Unknown target labels found: {unknown}")

print("Target distribution:")
print(y.value_counts(normalize=True))

Target distribution:
Rain
0    0.8744
1    0.1256
Name: proportion, dtype: float64


## 4. Train/Validation/Test Split
Split the dataset into train/validation/test sets with a fixed random seed and stratification.

In [4]:
%pip install scikit-learn matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (1750, 5) Val: (375, 5) Test: (375, 5)


## 5. Train AdaBoost Classifier
Fit an AdaBoost classifier and evaluate performance on the test set.

In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

model = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42,
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, preds),
    "precision": precision_score(y_test, preds),
    "recall": recall_score(y_test, preds),
    "f1": f1_score(y_test, preds),
}
if hasattr(model, "predict_proba"):
    proba = model.predict_proba(X_test)[:, 1]
    metrics["roc_auc"] = roc_auc_score(y_test, proba)
else:
    metrics["roc_auc"] = float("nan")
metrics

{'accuracy': 0.9973333333333333,
 'precision': 1.0,
 'recall': 0.9787234042553191,
 'f1': 0.989247311827957,
 'roc_auc': 0.9991242864556305}

## 6. Save the Model
Persist the trained model to disk for use in the Streamlit app.

In [7]:
import joblib

joblib.dump(model, "adaboost_rain_model.pkl")
print("Saved model to adaboost_rain_model.pkl")

Saved model to adaboost_rain_model.pkl
